In [1]:
pip install streamlit

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 3.2 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 3.1 MB/s  0:00:00 eta 0:00:01
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
pip install streamlit-option-menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 2.2 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Model Deployment

## Objective

In this notebook, we will validate the complete deployment pipeline.

The trained model, preprocessing pipeline, and label encoder will be loaded from the saved artifacts.

We will:

- Load deployment artifacts
- Load processed test data
- Perform single prediction
- Compare prediction with actual label
- Predict class probabilities
- Perform batch predictions
- Validate the deployment pipeline before building the Streamlit application

At the end of this notebook, we will have a production-ready prediction workflow.

In [12]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd

from pathlib import Path
import sys

In [13]:
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [14]:
from src.data.dataset_manager import DatasetManager

In [15]:
model = joblib.load(
    "../artifacts/best_model.pkl"
)

preprocessor = joblib.load(
    "../artifacts/preprocessor.pkl"
)

label_encoder = joblib.load(
    "../artifacts/label_encoder.pkl"
)

print("✅ Artifacts Loaded Successfully")

✅ Artifacts Loaded Successfully


In [16]:
dataset_manager = DatasetManager()

X_train_processed, X_test_processed, y_train, y_test = (
    dataset_manager.load_processed_data()
)

print(X_test_processed.shape)

✅ Processed datasets loaded successfully.
(138018, 25)


In [17]:
sample = X_test_processed.head(1)

display(sample)

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type_balanced,diet_type_non-veg,diet_type_veg,...,sleep_quality_poor,physical_activity_level_active,physical_activity_level_moderate,physical_activity_level_sedentary,smoking_alcohol_no,smoking_alcohol_occasional,smoking_alcohol_yes,gender_female,gender_male,gender_other
0,-0.150712,-0.885602,1.7043,0.708761,1.243157,0.077354,-0.273629,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [18]:
actual_label = y_test.iloc[0]

print("Actual Label :", actual_label)

Actual Label : at-risk


In [19]:
prediction = model.predict(sample)

predicted_label = label_encoder.inverse_transform(
    prediction
)

print("Predicted Label :", predicted_label[0])

Predicted Label : at-risk


In [20]:
probability = model.predict_proba(sample)

probability_df = pd.DataFrame(

    probability,

    columns=label_encoder.classes_

)

display(probability_df)

,at-risk,fit,unhealthy
0,0.998604,0.000017,0.00138


In [21]:
probability = model.predict_proba(sample)

probability_df = pd.DataFrame(
    probability,
    columns=label_encoder.classes_
)

display(probability_df)

predicted_class = probability_df.idxmax(axis=1).iloc[0]
confidence = probability_df.max(axis=1).iloc[0] * 100

print(f"Predicted Class : {predicted_class}")
print(f"Confidence      : {confidence:.2f}%")

,at-risk,fit,unhealthy
0,0.998604,0.000017,0.00138


Predicted Class : at-risk
Confidence      : 99.86%


In [22]:
actual = y_test.iloc[0]

comparison = pd.DataFrame({
    "Actual": [actual],
    "Predicted": [predicted_class],
    "Confidence (%)": [round(confidence, 2)]
})

display(comparison)

,Actual,Predicted,Confidence (%)
0,at-risk,at-risk,99.86


In [23]:
batch = X_test_processed.head(10)

predictions = model.predict(batch)

decoded_predictions = label_encoder.inverse_transform(predictions)

probabilities = model.predict_proba(batch)

confidence_scores = probabilities.max(axis=1) * 100

results = pd.DataFrame({
    "Actual": y_test.head(10).values,
    "Predicted": decoded_predictions,
    "Confidence (%)": confidence_scores.round(2)
})

display(results)

,Actual,Predicted,Confidence (%)
0,at-risk,at-risk,99.86
1,at-risk,at-risk,98.30
2,at-risk,at-risk,99.40
3,at-risk,at-risk,98.08
4,at-risk,at-risk,99.53
5,at-risk,at-risk,99.09
6,at-risk,at-risk,99.61
7,at-risk,at-risk,98.62
8,at-risk,at-risk,99.36
9,at-risk,at-risk,99.08


In [24]:
from sklearn.metrics import accuracy_score

full_predictions = model.predict(X_test_processed)

deployment_accuracy = accuracy_score(
    label_encoder.transform(y_test),
    full_predictions
)

print(f"Deployment Accuracy : {deployment_accuracy:.4f}")

Deployment Accuracy : 0.9657


In [26]:
from src.pipeline.prediction_pipeline import PredictionPipeline

pipeline = PredictionPipeline()

In [29]:
import pandas as pd

sample = pd.DataFrame({

    "sleep_duration":[7.5],

    "heart_rate":[70],

    "bmi":[23],

    "calorie_expenditure":[2200],

    "step_count":[9000],

    "exercise_duration":[45],

    "water_intake":[2.5],

    "diet_type":["balanced"],

    "stress_level":["low"],

    "sleep_quality":["good"],

    "physical_activity_level":["active"],

    "smoking_alcohol":["no"],

    "gender":["male"]

})

In [30]:
result = pipeline.get_prediction(sample)

print(result)

{'prediction': 'fit', 'probability': array([7.72807989e-03, 9.91482195e-01, 7.89725521e-04]), 'confidence': np.float64(0.991482194585794)}
